In [47]:
#Data manipulation
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
train_test_split,
RandomizedSearchCV,
TimeSeriesSplit,
cross_val_score
)

# K-mean clustering
from sklearn.cluster import KMeans

# Classifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (precision_recall_curve,
roc_curve,
RocCurveDisplay,
ConfusionMatrixDisplay
)

from sklearn.metrics import (accuracy_score,
f1_score,
recall_score,
precision_score,
roc_auc_score,
auc
)

from sklearn.metrics import (classification_report,
confusion_matrix
)

# Import tensorflow and keras
import tensorflow
import keras 

# Import pandas-ta
import pandas_ta as ta


#### Retrieving the Historical Return 

ใช้ข้อมูลเริ่มต้นปี 2014

In [59]:
import pandas as pd

files = ["CMA.csv", "HML.csv", "MKT.csv", "Return.csv", "RMW.csv", "SMB.csv", "UMD.csv"]
tickers = ["cma", "hml", "mkt", "Return", "rmw", "smb", "umd"]

data = pd.DataFrame()

# Load files and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    
    # Remove duplicate indices if any
    df = df[~df.index.duplicated(keep='first')]
    
    # Add suffix "_close" to column names
    if len(df.columns) > 1:
        df.columns = [f"{col}_{tickers[i]}_close" for col in df.columns]
    else:
        df.columns = [f"{tickers[i]}_close"]
    
    # Concatenate data
    data = pd.concat([data, df], axis=1)

# Save each original ticker data into a single dataframe with "_close" suffix
for ticker in tickers:
    df = pd.read_csv(ticker + ".csv", index_col=0, parse_dates=True)
    
    # Remove duplicate indices if any
    df = df[~df.index.duplicated(keep='first')]
    
    # Add suffix "_close" to column name
    df.columns = [f"{ticker}_Close"]
    
    globals()[ticker + "_Close"] = df  # Create a global variable with "_close" suffix

# Display the combined DataFrame with "_close" suffix added
print(data.head())

          cma_close  hml_close  mkt_close  Return_close  rmw_close  smb_close  \
date                                                                            
2/1/2014   0.005573   0.016541  -0.053221     -0.047340   0.002402   0.012353   
3/1/2014  -0.001040   0.016333  -0.005285      0.007205  -0.006938  -0.007504   
6/1/2014  -0.000661  -0.008409   0.005480      0.006666  -0.008829  -0.018920   
7/1/2014  -0.006445  -0.015975   0.026072      0.020988   0.007696  -0.004548   
8/1/2014   0.003194  -0.005862  -0.003476     -0.003139   0.003944  -0.003521   

          umd_close  
date                 
2/1/2014  -0.013186  
3/1/2014  -0.001181  
6/1/2014   0.011955  
7/1/2014   0.005953  
8/1/2014  -0.001385  


In [61]:
import pandas as pd

files = ["CMA.csv", "HML.csv", "MKT.csv", "Return.csv", "RMW.csv", "SMB.csv", "UMD.csv"]
tickers = ["cma", "hml", "mkt", "Return", "rmw", "smb", "umd"]

data = pd.DataFrame()

# Load files and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    
    # Remove duplicate indices if any
    df = df[~df.index.duplicated(keep='first')]
    
    # Add suffixes to columns
    df.columns = [f"{col}_{tickers[i]}" for col in df.columns]
    
    # Concatenate data
    data = pd.concat([data, df], axis=1)

# Also save each original ticker data into a single dataframe for later calculation of technical indicators
for ticker in tickers:
    df = pd.read_csv(ticker + ".csv", index_col=0, parse_dates=True)
    
    # Remove duplicate indices if any
    df = df[~df.index.duplicated(keep='first')]
    
    globals()[ticker] = df

data

,cma_cma,hml_hml,mkt_mkt,Return_Return,rmw_rmw,smb_smb,umd_umd
date,,,,,,,
2/1/2014,0.005573,0.016541,-0.053221,-0.047340,0.002402,0.012353,-0.013186
3/1/2014,-0.001040,0.016333,-0.005285,0.007205,-0.006938,-0.007504,-0.001181
6/1/2014,-0.000661,-0.008409,0.005480,0.006666,-0.008829,-0.018920,0.011955
7/1/2014,-0.006445,-0.015975,0.026072,0.020988,0.007696,-0.004548,0.005953
8/1/2014,0.003194,-0.005862,-0.003476,-0.003139,0.003944,-0.003521,-0.001385
...,...,...,...,...,...,...,...
24/4/2024,-0.006187,-0.010302,0.003068,-0.002653,0.003958,-0.005578,0.005125
25/4/2024,-0.001990,-0.004040,0.002615,0.004100,0.002175,0.003942,0.002862
26/4/2024,-0.007605,0.007793,-0.003100,-0.003670,0.004133,0.002374,0.000403


In [49]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
cma_close,2507.0,0.000213,0.005698,-0.047966,-0.002797,0.000223,0.003252,0.046206
hml_close,2507.0,0.000146,0.005852,-0.022967,-0.003359,-0.000034,0.003567,0.028299
mkt_close,2507.0,0.000142,0.009317,-0.108409,-0.004124,0.000333,0.004660,0.082148
Return_close,2507.0,-0.000699,0.007273,-0.058480,-0.004498,-0.000604,0.003069,0.119884
rmw_close,2507.0,0.000032,0.004787,-0.026652,-0.002879,0.000003,0.002987,0.025009
smb_close,2507.0,0.000083,0.006097,-0.052801,-0.003004,0.000281,0.003631,0.038111
umd_close,2507.0,0.000284,0.007424,-0.100241,-0.003617,0.000217,0.004275,0.057125


In [55]:
# Create your own Custom Strategy
CustomStrategy = ta.Strategy(
    name="Momo and Volatility",
    description="SMA 50,200, BBANDS, RSI, MACD and Volume SMA 20",
    ta=[
        {"kind": "sma"},
        {"kind": "ema"},
        {"kind": "roc"},
        {"kind": "rsi"},
        {"kind": "bbands"},
        {"kind": "macd"},
        {"kind": "stoch"},
        {"kind": "swma"},
    ]
)

# To run your "Custom Strategy"
df.ta.strategy(CustomStrategy)

In [60]:
# Calculate all technical indicators for cma
df = cma
ind1 = df.ta.strategy(CustomStrategy, df,)
ind1 = df.drop(df.iloc[:, 0:6], axis=1)
ind1

,EMA_20_cma,cma_cma,SMA_10_cma,ROC_cma,RSI_cma,EMA_10_cma_cma,EMA_20_cma_cma,cma_cma_cma,SMA_10_cma_cma,ROC_cma_cma,...,ROC_cma_cma_cma_cma_cma_cma_cma,RSI_cma_cma_cma_cma_cma_cma_cma,EMA_10_cma_cma_cma_cma_cma_cma_cma_cma,EMA_20_cma_cma_cma_cma_cma_cma_cma_cma,cma_cma_cma_cma_cma_cma_cma_cma_cma,SMA_10_cma_cma_cma_cma_cma_cma_cma_cma,ROC_cma_cma_cma_cma_cma_cma_cma_cma,RSI_cma_cma_cma_cma_cma_cma_cma_cma,EMA_10_cma_cma_cma_cma_cma_cma_cma_cma_cma,EMA_20_cma_cma_cma_cma_cma_cma_cma_cma_cma
date,,,,,,,,,,,,,,,,,,,,,
2/1/2014,NaN,0.005573,NaN,NaN,NaN,NaN,NaN,0.005573,NaN,NaN,...,NaN,NaN,NaN,NaN,0.005573,NaN,NaN,NaN,NaN,NaN
3/1/2014,NaN,-0.001040,NaN,NaN,NaN,NaN,NaN,-0.001040,NaN,NaN,...,NaN,NaN,NaN,NaN,-0.001040,NaN,NaN,NaN,NaN,NaN
6/1/2014,NaN,-0.000661,NaN,NaN,NaN,NaN,NaN,-0.000661,NaN,NaN,...,NaN,NaN,NaN,NaN,-0.000661,NaN,NaN,NaN,NaN,NaN
7/1/2014,NaN,-0.006445,NaN,NaN,NaN,NaN,NaN,-0.006445,NaN,NaN,...,NaN,NaN,NaN,NaN,-0.006445,NaN,NaN,NaN,NaN,NaN
8/1/2014,NaN,0.003194,NaN,NaN,NaN,NaN,NaN,0.003194,NaN,NaN,...,NaN,NaN,NaN,NaN,0.003194,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24/4/2024,0.000484,-0.006187,0.000344,-257.024756,45.530290,4.416386e-04,0.000484,-0.006187,0.000344,-257.024756,...,-257.024756,45.530290,4.416386e-04,0.000484,-0.006187,0.000344,-257.024756,45.530290,4.416386e-04,0.000484
25/4/2024,0.000249,-0.001990,0.000147,9605.336585,48.442245,-4.036559e-07,0.000249,-0.001990,0.000147,9605.336585,...,9605.336585,48.442245,-4.036559e-07,0.000249,-0.001990,0.000147,9605.336585,48.442245,-4.036559e-07,0.000249
26/4/2024,-0.000499,-0.007605,0.000301,-16.811955,44.978209,-1.383002e-03,-0.000499,-0.007605,0.000301,-16.811955,...,-16.811955,44.978209,-1.383002e-03,-0.000499,-0.007605,0.000301,-16.811955,44.978209,-1.383002e-03,-0.000499
